In [34]:
import pandas as pd
import os
import kagglehub
from dotenv import load_dotenv
import numpy as np

In [35]:
#Load .env file for Kaggle Credentials 
load_dotenv()
username = os.getenv("KAGGLE_USERNAME")
key = os.getenv("KAGGLE_KEY")

os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = key

In [36]:
#Locate path to the Kaggle dataset
path = kagglehub.dataset_download(
    "aiaiaidavid/the-big-dataset-of-ultra-marathon-running"
)

In [37]:
df = pd.read_csv(f"{path}/TWO_CENTURIES_OF_UM_RACES.csv", encoding="latin1")
df = df.drop(columns=["Event dates", "Athlete club", "Event number of finishers", "Athlete average speed"])

KeyboardInterrupt: 

In [ ]:
#1) Extract data since 2000s (TRY DIFFERENT THRESHOLDS)
df = df[df['Year of event'] >= 2000]

In [ ]:
#2) Extract athletes that have data on a large portion of their career
threshold_of_races = 10
valid_athletes = df['Athlete ID'].value_counts()
valid_athletes = valid_athletes[valid_athletes > threshold_of_races].index
experienced_df = df[df['Athlete ID'].isin(valid_athletes)]
experienced_df['Athlete ID'].nunique()

142777

In [ ]:
#Feature of race_count --> Number of times a unique ID appears
experienced_df = df[df['Athlete ID'].isin(valid_athletes)].copy()
experienced_df["race_count"] = experienced_df.groupby("Athlete ID")["Athlete ID"].transform("count")

In [ ]:
#Count the number of entries for each race
experienced_df['Event name'].value_counts()

Event name
Two Oceans Marathon (RSA)                                                  88506
Comrades Marathon - Down Run (RSA)                                         69850
Comrades Marathon - Up Run (RSA)                                           64314
Two Oceans Marathon - 50km Split (RSA)                                     50642
Ultra Trail Tour du Mont Blanc (UTMB) (FRA)                                19693
                                                                           ...  
SzÅlÅsKÃ¶rÃzgetÅs&KarÃ¡csonyi UltrAngyalok - 12h - 6h Split (HUN)          1
SzÅlÅsKÃ¶rÃzgetÅs&KarÃ¡csonyi UltrAngyalok - 12h - 50mi Split (HUN)        1
100 Milhas De SabarabuÃ§u - 90 Kms (BRA)                                       1
Buckley's Chance 50K Trail Run (AUS)                                           1
UNLEASHED - A ThunderMeet Event 6 Hour Race (USA)                              1
Name: count, Length: 24004, dtype: int64

In [ ]:
df_two_oceans = experienced_df[experienced_df['Event name'] == 'Two Oceans Marathon (RSA)'].copy()
df_two_oceans.dropna()
print(f"Total athletes: {df_two_oceans["Athlete ID"].nunique()}")
print(f"Total entries: {len(df_two_oceans["Athlete ID"])}")

Total athletes: 16711
Total entries: 88506


In [ ]:
#Feature two_oceans_count --> Number of times they have ran this event
df_two_oceans["two_oceans_count"] = df_two_oceans.groupby("Athlete ID")["Athlete ID"].transform("count")
df_two_oceans["two_oceans_count"]

171455      7
171456      5
171458      9
171461      2
171462      9
           ..
6360639    14
6360643     3
6360647     3
6360648     7
6360649     4
Name: two_oceans_count, Length: 88506, dtype: int64

In [ ]:
#Calculate the athletes average speed in km/hr --> Used to Extract top threshold of fastest runners
#Since format H:MM:SS h harder to compare, numerically

df_two_oceans["distance_km"] = df_two_oceans["Event distance/length"].str.extract(r"(\d+\.?\d*)").astype(float)
df_two_oceans["distance_km"]
time_parts = df_two_oceans["Athlete performance"].str.replace(" h", "").str.split(":", expand=True)

df_two_oceans["hours"] = (
    time_parts[0].astype(float) +
    time_parts[1].astype(float) / 60 +
    time_parts[2].astype(float) / 3600
)

df_two_oceans["calculated_speed"] = df_two_oceans["distance_km"] / df_two_oceans["hours"]
df_two_oceans["calculated_speed"].head()

171455    17.634710
171456    17.571690
171458    17.486339
171461    17.297297
171462    17.241084
Name: calculated_speed, dtype: float64

In [ ]:
#Filter the top 10% of the fastest athletes
# adding feature for 'elite' runners, top 10% threshold can be adjusted
# df_two_oceans['Elite'] = df_two_oceans['Calculated speed'] >= df_two_oceans['Calculated speed'].quantile(0.95)  # per race or per year
threshold = df_two_oceans["calculated_speed"].quantile(0.90)

df_top10 = df_two_oceans[df_two_oceans["calculated_speed"] >= threshold].copy()

df_top10["finish_time_mins"] = df_top10["hours"] * 60

print(f"Total Entries: {len(df_two_oceans)}")
print(f"Top 10% Entries: {len(df_top10)}\n")
print(df_top10["calculated_speed"].describe())


Total Entries: 88506
Top 10% Entries: 8853

count    8853.000000
mean       13.642159
std         1.195476
min        12.234494
25%        12.673666
50%        13.331570
75%        14.271556
max        18.096948
Name: calculated_speed, dtype: float64


In [ ]:
# runners are overwhelmingly from South Africa for this race
# potential domestic/foreign feature??
df_two_oceans["Athlete country"].value_counts()

Athlete country
RSA    84091
GBR     1024
GER      607
ZIM      462
NAM      384
       ...  
XXX        1
PHI        1
ESP        1
LAT        1
ROU        1
Name: count, Length: 68, dtype: int64

In [ ]:
# 1. Athlete Age
df_top10["athlete_age"] = df_top10["Year of event"] - df_top10["Athlete year of birth"]

# 2. Age squared - captures non-linear age effect (performance peaks then declines)
df_top10["athlete_age_squared"] = df_top10["athlete_age"] ** 2

# 3. Gender - binary encode
df_top10["gender_male"] = (df_top10["Athlete gender"] == "M").astype(int)

# 4. Domestic vs International
df_top10["is_domestic"] = (df_top10["Athlete country"] == "RSA").astype(int)

# 5. Age from peak (marathon peak age ~28-32, using 30 as midpoint)
df_top10["age_from_peak"] = abs(df_top10["athlete_age"] - 30)

# 6. Experience - log transform to diminish returns of extra races
# log(1 + x) ensures athletes with 0 previous races don't cause log(0)
df_top10["experience_log"] = np.log1p(df_top10["race_count"] * df_top10["two_oceans_count"])

# 7. Age x Gender interaction
df_top10["age_gender"] = df_top10["athlete_age"] * df_top10["gender_male"]

# 8. Age x log(experience) interaction
df_top10["age_x_experience"] = df_top10["athlete_age"] * df_top10["experience_log"]

In [38]:
features = [
    "athlete_age",
    "athlete_age_squared",
    "gender_male",
    "is_domestic",
    "age_from_peak",
    "experience_log",
    "age_gender",
    "age_x_experience"
]

target = "finish_time_mins"

# 1. Correlation with TARGET
target_corr = df_top10[features + ["finish_time_mins"]].corr()["finish_time_mins"].sort_values()
print(target_corr)

# 2. Feature-to-feature correlation — multicollinearity check  
feature_corr = df_top10[features].corr()
print(feature_corr)

gender_male           -0.070146
experience_log         0.005100
age_gender             0.087148
age_x_experience       0.149215
is_domestic            0.190037
athlete_age            0.233966
athlete_age_squared    0.234403
age_from_peak          0.234916
finish_time_mins       1.000000
Name: finish_time_mins, dtype: float64
                     athlete_age  athlete_age_squared  gender_male  \
athlete_age             1.000000             0.992574     0.020012   
athlete_age_squared     0.992574             1.000000     0.023911   
gender_male             0.020012             0.023911     1.000000   
is_domestic            -0.020459            -0.021459     0.090803   
age_from_peak           0.949249             0.968021     0.025372   
experience_log          0.184272             0.184655     0.036092   
age_gender              0.625665             0.625199     0.781654   
age_x_experience        0.741896             0.740861     0.036276   

                     is_domestic  age_from

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor 

In [40]:
features = [
    "gender_male",
    "is_domestic",
    "age_from_peak",
    "experience_log",
    "age_x_experience"
]

target = "finish_time_mins"

model_df = df_top10[features + [target, "Year of event"]].dropna()
model_df

,gender_male,is_domestic,age_from_peak,experience_log,age_x_experience,finish_time_mins,Year of event
171455,1,0,6.0,4.663439,167.883807,190.533333,2018
171456,1,1,7.0,5.049856,186.844672,191.216667,2018
171458,1,1,5.0,5.983936,209.437770,192.150000,2018
171461,1,0,0.0,3.367296,101.018875,194.250000,2018
171462,1,0,7.0,5.093750,188.468757,194.883333,2018
...,...,...,...,...,...,...,...
6352507,1,1,18.0,4.290459,205.942053,274.150000,2015
6352511,0,1,10.0,5.459586,218.383421,274.450000,2015
6352512,1,1,29.0,7.139660,421.239960,274.600000,2015
6352513,1,1,8.0,4.330733,164.567867,274.600000,2015


In [41]:
#TIME BASED MODEL SPLIT
train_df = model_df[model_df["Year of event"] <= 2015]
val_df   = model_df[(model_df["Year of event"] >= 2016) & (model_df["Year of event"] <= 2017)]
test_df  = model_df[model_df["Year of event"] >= 2018]

In [42]:
X_train, y_train = train_df[features], train_df[target]
X_val,   y_val   = val_df[features],   val_df[target]
X_test,  y_test  = test_df[features],  test_df[target]

In [43]:
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 7303 | Val: 800 | Test: 750


In [44]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform
X_val_s   = scaler.transform(X_val)         # transform only
X_test_s  = scaler.transform(X_test)        # transform only

In [45]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2   = r2_score(y_true, y_pred)
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  MAE:  {mae:.2f} mins")
    print(f"  RMSE: {rmse:.2f} mins")
    print(f"  R²:   {r2:.4f}")
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

results = []

In [46]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 1 — LINEAR REGRESSION (Ridge for stability)
# ══════════════════════════════════════════════════════════════════════════════
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, y_train)

# Tune alpha on validation set
best_alpha, best_val_mae = 1.0, float("inf")
for alpha in [0.01, 0.1, 1, 10, 100]:
    m = Ridge(alpha=alpha).fit(X_train_s, y_train)
    mae = mean_absolute_error(y_val, m.predict(X_val_s))
    if mae < best_val_mae:
        best_val_mae, best_alpha = mae, alpha

ridge_final = Ridge(alpha=best_alpha).fit(X_train_s, y_train)
results.append(evaluate("Ridge Regression", y_test, ridge_final.predict(X_test_s)))

# Coefficients — interpretability
coef_df = pd.DataFrame({"feature": features, "coef": ridge_final.coef_})
print(coef_df.sort_values("coef", key=abs, ascending=False))


────────────────────────────────────────
  Ridge Regression
  MAE:  14.93 mins
  RMSE: 18.27 mins
  R²:   0.0982
            feature      coef
1       is_domestic  4.281762
4  age_x_experience  3.638896
3    experience_log -3.266843
2     age_from_peak  2.790137
0       gender_male -1.950856


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 2 — REGRESSION TREE
# ══════════════════════════════════════════════════════════════════════════════
import itertools
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

param_grid = {
    "max_depth":        [2, 3, 4, 5, 6, 7, 8, None],  # None = fully grown
    "min_samples_split":[2, 5, 10, 20, 50],     # min samples to split a node
    "min_samples_leaf": [1, 5, 10, 20, 30],     # min samples at a leaf
    "max_features":     [None, "sqrt", "log2", 0.5, 0.7],  # features to consider per split
    "ccp_alpha":        [0.0, 0.001, 0.01, 0.1],  # pruning strength
}

keys, values = zip(*param_grid.items())
best_params, best_val_mae = {}, float("inf")

for combo in itertools.product(*values):
    params = dict(zip(keys, combo))
    m = DecisionTreeRegressor(**params, random_state=42).fit(X_train, y_train)
    mae = mean_absolute_error(y_val, m.predict(X_val))
    if mae < best_val_mae:
        best_val_mae, best_params = mae, params

print(f"Best val MAE: {best_val_mae:.2f}")
print(f"Best params: {best_params}")

tree_final = DecisionTreeRegressor(**best_params, random_state=42)
tree_final.fit(X_train, y_train)
results.append(evaluate(f"Regression Tree (tuned)", y_test, tree_final.predict(X_test)))

# Feature importances
imp_df = pd.DataFrame({"feature": features, "importance": tree_final.feature_importances_})
print(imp_df.sort_values("importance", ascending=False))

Best val MAE: 14.78
Best params: {'max_depth': 7, 'min_samples_split': 2, 'min_samples_leaf': 20, 'max_features': 0.7, 'ccp_alpha': 0.0}

────────────────────────────────────────
  Regression Tree (tuned)
  MAE:  14.92 mins
  RMSE: 18.23 mins
  R²:   0.1019
            feature  importance
2     age_from_peak    0.410512
1       is_domestic    0.248162
3    experience_log    0.198688
4  age_x_experience    0.071513
0       gender_male    0.071126


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 3 — GRADIENT BOOSTING (sklearn GBM)
# ══════════════════════════════════════════════════════════════════════════════
# Tune n_estimators & learning_rate on validation set
best_params, best_val_mae = {}, float("inf")
for n_est in [100, 200, 300]:
    for lr in [0.01, 0.05, 0.1]:
        m = GradientBoostingRegressor(
            n_estimators=n_est, learning_rate=lr,
            max_depth=4, subsample=0.8, random_state=42
        ).fit(X_train, y_train)
        mae = mean_absolute_error(y_val, m.predict(X_val))
        if mae < best_val_mae:
            best_val_mae, best_params = mae, {"n_estimators": n_est, "learning_rate": lr}

gbm_final = GradientBoostingRegressor(**best_params, max_depth=4, subsample=0.8, random_state=42)
gbm_final.fit(X_train, y_train)
results.append(evaluate(f"Gradient Boosting {best_params}", y_test, gbm_final.predict(X_test)))


────────────────────────────────────────
  Gradient Boosting {'n_estimators': 200, 'learning_rate': 0.1}
  MAE:  14.84 mins
  RMSE: 18.26 mins
  R²:   0.0991
